# Prepare OCR Text

In [1]:
import os
import json

input_folder = r"..\data_ocr\test\Text"
output_folder = r"..\data_ocr\test\Lines"

os.makedirs(output_folder, exist_ok=True)

In [2]:
# Take the text content only and save in Lines folder

for filename in os.listdir(input_folder):
    if filename.endswith(".json"):
        input_path = os.path.join(input_folder, filename)
        with open(input_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # extract text lines
        lines = [item["text"].strip() for item in data]
        output_path = os.path.join(output_folder, filename)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(lines, f, indent=4, ensure_ascii=False)
print("Conversion complete.")

Conversion complete.


# LLM Prompt

In [ ]:
import sys
!{sys.executable} -m pip install transformers==4.41.2

In [ ]:
import sys
!{sys.executable} -m pip install ipywidgets

In [1]:
import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# load model
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [15]:
prompt = """
Extract the company, date, and total.

Receipt:
KFC receipt
Date 12/05/2022
Total 45.30

Answer in JSON form
"""
print(pipe(prompt, max_new_tokens=100))

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '\nExtract the company, date, and total.\n\nReceipt:\nKFC receipt\nDate 12/05/2022\nTotal 45.30\n\nAnswer in JSON form\n'}]


In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

In [2]:
import json

# read OCR json file
with open(r"D:\sroie-ocr-information-extraction\data_ocr\test\Lines\000.json", "r", encoding="utf-8") as f:
    ocr_lines = json.load(f)
with open(r"D:\sroie-ocr-information-extraction\data_ocr\test\Lines\001.json", "r", encoding="utf-8") as f:
    ocr_lines1 = json.load(f)
# convert list -> text
ocr_text = "\n".join(ocr_lines)
ocr_text1 = "\n".join(ocr_lines1)

# build prompt
prompt = f"""
Given an example: 
Extracting information from text:
{ocr_text1}
and get the result:
    "company": "OJC MARKETING SDN BHD",
    "date": "02/01/2019",
    "address": "NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR",
    "total": "170.00"

Do exactly with this text:
{ocr_text}
"""

# tokenize
inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

# generate answer
outputs = model.generate(**inputs, max_new_tokens=100)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(result)

***COPY*** OJC MARKETING SDN BHD ROC NO:538358-H NO2&4JALANBAYU4, BANDAR SERI ALAM, 81750MASAI,JOHOR Tel:07-388 2218 Fax:07-3888218 Email:ng@ojcgroup.com Cash Bill InvoiceNoPEGIV-1030531 Date 02/01/20192:47:14PM Cashier R


In [17]:
import json

# read OCR json file
with open(r"D:\sroie-ocr-information-extraction\data_ocr\test\Lines\000.json", "r", encoding="utf-8") as f:
    ocr_lines = json.load(f)
with open(r"D:\sroie-ocr-information-extraction\data_ocr\test\Lines\001.json", "r", encoding="utf-8") as f:
    ocr_lines1 = json.load(f)
with open(r"D:\sroie-ocr-information-extraction\data\test\entities\001.json", "r", encoding="utf-8") as f:
    key = json.load(f)
# convert list -> text
ocr_text = " ".join(ocr_lines)
ocr_text1 = " ".join(ocr_lines1)
#key = "\n".join(key_lines)

# build prompt
prompt = f"""
Extract information from the receipt OCR text.

Return ONLY a JSON object.

Fields:
company
date
address
total

Example:

OCR:
{ocr_text1}

Output:
{key}

OCR:
{ocr_text}
Output:
    "company": 
    "date":    
    "address":
    "total":
"""

# tokenize
inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

# generate answer
outputs = model.generate(**inputs, max_new_tokens=100)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(result)

KINGS SAFETY SHO


In [16]:
key

{'company': 'OJC MARKETING SDN BHD',
 'date': '02/01/2019',
 'address': 'NO 2 & 4, JALAN BAYU 4, BANDAR SERI ALAM, 81750 MASAI, JOHOR',
 'total': '170.00'}

In [12]:
key = "\n".join(key_lines)
key

'company\ndate\naddress\ntotal'

In [3]:
import re

def normalize(text):
    text = text.upper()
    text = re.sub(r'[^A-Z0-9]', '', text)
    return text

In [4]:
res = normalize(result)

Download model

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "microsoft/Phi-3-mini-4k-instruct"

save_path = "../models/phi3-mini"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

tokenizer.save_pretrained(save_path)
model.save_pretrained(save_path)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

C:\Users\hana2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hana2\.cache\huggingface\hub\models--microsoft--Phi-3-mini-4k-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

: 